# 07 — Reaction Energetics: Enthalpies, Isodesmic Reactions & Bond Dissociation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/07_reaction_energetics.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Compute reaction enthalpies from electronic energies and ZPE corrections
- Understand and design isodesmic reactions to cancel systematic errors
- Calculate bond dissociation energies (BDEs) for homolytic cleavage
- Compute atomization energies and heats of formation
- Critically compare DFT reaction energies with experimental thermochemical data
- Recognize when systematic error cancellation improves accuracy

## 1. Theory: Reaction Energetics

### 1.1 Reaction Energy

For a balanced reaction $\sum_i \nu_i \text{A}_i = 0$ (products have $\nu > 0$, reactants $\nu < 0$):

$$\Delta E_{rxn} = \sum_{\text{products}} E_i - \sum_{\text{reactants}} E_j$$

With zero-point energy correction:

$$\Delta E_{rxn}^{ZPE} = \Delta E_{rxn} + \Delta ZPE$$

$$\Delta ZPE = \frac{1}{2} \hbar \left(\sum_{\text{products}} \omega_k - \sum_{\text{reactants}} \omega_k\right)$$

### 1.2 Isodesmic Reactions

An **isodesmic reaction** conserves the number and type of bonds on both sides. Because the same bond types cancel, systematic DFT errors largely cancel:

$$\text{(strained molecule)} + n\,\text{(reference)} \longrightarrow \text{(open-chain fragments)}$$

For cyclopropane strain energy:
$$\text{C}_3\text{H}_6 + 3\,\text{CH}_4 \longrightarrow 3\,\text{C}_2\text{H}_6$$

The strain energy $E_{strain}$ is extracted from the isodesmic $\Delta E$ plus known experimental heats of formation.

### 1.3 Bond Dissociation Energy

The **homolytic BDE** for A–B cleavage:

$$\text{BDE}(\text{A-B}) = E(\text{A}^\bullet) + E(\text{B}^\bullet) - E(\text{A-B})$$

Note: radicals ($\text{A}^\bullet$, $\text{B}^\bullet$) require **unrestricted** (UKS/UHF) calculations.

### 1.4 Heats of Formation

From the **atomization energy** approach:

$$\Delta H_f^\circ(\text{molecule}) = \Delta H_{at}^{\text{DFT}} - \sum_A \Delta H_{at,A}^{\text{DFT}} + \sum_A \Delta H_f^\circ(\text{atom A, expt})$$

This requires accurate atomic energies and corrections for spin–orbit coupling and the zero-point energy.

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 07: Reaction Energetics
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
from pyscf import gto, dft, scf

HA2KCAL = 627.509  # Hartree to kcal/mol

def run_rks(atom_str, basis='def2-SVP', xc='B3LYP', charge=0, spin=0, verbose=0):
    """Run RKS (closed-shell DFT) and return (mol, mf)."""
    mol = gto.Mole(atom=atom_str, basis=basis, charge=charge, spin=spin, verbose=verbose)
    mol.build()
    mf = dft.RKS(mol) if spin == 0 else dft.UKS(mol)
    mf.xc = xc
    mf.verbose = verbose
    mf.kernel()
    return mol, mf

# ------------------------------------------------------------------
# Reaction 1: H2 + F2 → 2 HF
# ------------------------------------------------------------------
_, mf_h2  = run_rks('H 0 0 0; H 0 0 0.74')
_, mf_f2  = run_rks('F 0 0 0; F 0 0 1.41')
_, mf_hf  = run_rks('H 0 0 0; F 0 0 0.92')

dE_rxn1 = 2*mf_hf.e_tot - (mf_h2.e_tot + mf_f2.e_tot)
print('Reaction: H₂ + F₂ → 2 HF')
print(f'  ΔE_rxn = {dE_rxn1*HA2KCAL:.2f} kcal/mol  (expt: -128.6 kcal/mol)')

# ------------------------------------------------------------------
# Reaction 2: H2O formation  N2 + 3H2 -> 2NH3 as well
# ------------------------------------------------------------------
_, mf_o2  = run_rks('O 0 0 0; O 0 0 1.21')
_, mf_h2o = run_rks('O 0 0 0.117; H 0  0.757 -0.469; H 0 -0.757 -0.469')

dE_rxn2 = 2*mf_h2o.e_tot - (2*mf_h2.e_tot + mf_o2.e_tot)
print('\nReaction: 2 H₂ + O₂ → 2 H₂O')
print(f'  ΔE_rxn = {dE_rxn2*HA2KCAL:.2f} kcal/mol  (expt: -241.8 kJ/mol per H₂O ≈ -115.6 kcal/mol per H₂O)')
print(f'  Per H₂O: {dE_rxn2/2*HA2KCAL:.2f} kcal/mol')

print('\nAll energies at B3LYP/def2-SVP')
print(f'  E(H₂)  = {mf_h2.e_tot:.6f} Ha')
print(f'  E(F₂)  = {mf_f2.e_tot:.6f} Ha')
print(f'  E(HF)   = {mf_hf.e_tot:.6f} Ha')
print(f'  E(O₂)  = {mf_o2.e_tot:.6f} Ha')
print(f'  E(H₂O) = {mf_h2o.e_tot:.6f} Ha')

In [ ]:
%%time
# ------------------------------------------------------------------
# Bond Dissociation Energies (BDEs)
# ------------------------------------------------------------------
# H2 BDE: H2 -> 2 H•  (homolytic)
# CH4 BDE: CH4 -> CH3• + H•
# Use UKS for open-shell radicals

from pyscf import dft

def run_uks(atom_str, basis='def2-SVP', xc='B3LYP', charge=0, spin=1, verbose=0):
    mol = gto.Mole(atom=atom_str, basis=basis, charge=charge, spin=spin, verbose=verbose)
    mol.build()
    mf = dft.UKS(mol)
    mf.xc = xc
    mf.verbose = verbose
    mf.kernel()
    return mol, mf

# H• (doublet, spin=1)
_, mf_h_rad  = run_uks('H 0 0 0', spin=1)

# H2 BDE
bde_h2 = (2*mf_h_rad.e_tot - mf_h2.e_tot) * HA2KCAL
print('BDE(H-H) = E(H•) + E(H•) - E(H₂)')
print(f'  BDE(H₂) = {bde_h2:.1f} kcal/mol  (expt: 104.2 kcal/mol)')

# CH4 geometry (Td)
ch4_geom = '''
C   0.000000   0.000000   0.000000
H   0.629118   0.629118   0.629118
H  -0.629118  -0.629118   0.629118
H  -0.629118   0.629118  -0.629118
H   0.629118  -0.629118  -0.629118
'''
_, mf_ch4 = run_rks(ch4_geom, spin=0)

# CH3• (doublet, C3v)
ch3_geom = '''
C   0.000000   0.000000   0.000000
H   1.079000   0.000000   0.000000
H  -0.539500   0.934200   0.000000
H  -0.539500  -0.934200   0.000000
'''
_, mf_ch3 = run_uks(ch3_geom, spin=1)

bde_ch4 = (mf_ch3.e_tot + mf_h_rad.e_tot - mf_ch4.e_tot) * HA2KCAL
print(f'\nBDE(C-H in CH₄) = E(CH₃•) + E(H•) - E(CH₄)')
print(f'  BDE(CH₄) = {bde_ch4:.1f} kcal/mol  (expt: 105.1 kcal/mol)')

# ------------------------------------------------------------------
# Summary table
# ------------------------------------------------------------------
bde_data = {
    'Bond': ['H–H (H₂)', 'C–H (CH₄)'],
    'Computed (kcal/mol)': [round(bde_h2, 1), round(bde_ch4, 1)],
    'Experimental (kcal/mol)': [104.2, 105.1],
    'Error (kcal/mol)': [round(bde_h2-104.2, 1), round(bde_ch4-105.1, 1)],
}
df_bde = pd.DataFrame(bde_data)
print('\nBDE Summary (B3LYP/def2-SVP):')
print(df_bde.to_string(index=False))

In [ ]:
%%time
# ------------------------------------------------------------------
# Isodesmic Reaction: Strain Energy of Cyclopropane
#
# C3H6 + 3 CH4 --> 3 C2H6
# Strain energy = -ΔE_isodesmic (literature: ~27.5 kcal/mol)
# ------------------------------------------------------------------

# Cyclopropane (D3h): C-C bond 1.513 A, C-H bond 1.083 A
cyc_geom = '''
C   0.000000   0.877000   0.000000
C   0.759000  -0.438500   0.000000
C  -0.759000  -0.438500   0.000000
H   0.000000   1.462000   0.904000
H   0.000000   1.462000  -0.904000
H   1.265000  -0.731000   0.904000
H   1.265000  -0.731000  -0.904000
H  -1.265000  -0.731000   0.904000
H  -1.265000  -0.731000  -0.904000
'''
_, mf_c3h6 = run_rks(cyc_geom, spin=0)

# Ethane (C2h staggered)
eth_geom = '''
C   0.000000   0.000000   0.765000
C   0.000000   0.000000  -0.765000
H   1.018000   0.000000   1.157000
H  -0.509000   0.882000   1.157000
H  -0.509000  -0.882000   1.157000
H  -1.018000   0.000000  -1.157000
H   0.509000   0.882000  -1.157000
H   0.509000  -0.882000  -1.157000
'''
_, mf_c2h6 = run_rks(eth_geom, spin=0)

dE_iso = (3*mf_c2h6.e_tot - (mf_c3h6.e_tot + 3*mf_ch4.e_tot)) * HA2KCAL
strain = -dE_iso
print('Isodesmic reaction: C₃H₆ + 3 CH₄ → 3 C₂H₆')
print(f'  ΔE_isodesmic = {dE_iso:.2f} kcal/mol')
print(f'  Strain energy = {strain:.2f} kcal/mol  (expt: ~27.5 kcal/mol)')
print(f'  Error = {strain - 27.5:.2f} kcal/mol')
print()

# ------------------------------------------------------------------
# Comparison plot
# ------------------------------------------------------------------
labels = ['H₂+F₂→2HF', '2H₂+O₂→2H₂O\n(per H₂O)', 'BDE(H₂)', 'BDE(CH₄)', 'Cyclopropane\nstrain']
computed = [dE_rxn1*HA2KCAL, dE_rxn2/2*HA2KCAL, bde_h2, bde_ch4, strain]
experimental = [-128.6, -57.8, 104.2, 105.1, 27.5]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, computed, width, label='B3LYP/def2-SVP', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, experimental, width, label='Experimental', color='coral', alpha=0.85)
ax.set_ylabel('Energy (kcal/mol)')
ax.set_title('Reaction Energies: B3LYP/def2-SVP vs Experiment')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend()
ax.axhline(0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 🔬 Research Connection

Reaction energetics underpin virtually all of computational chemistry:

- **Combustion chemistry**: BDEs of C–H, O–H, and N–H bonds determine fuel ignition and oxidation mechanisms. Isodesmic networks anchor computed values to ATcT (Active Thermochemical Tables).
- **Drug metabolism**: C–H BDEs in drug candidates predict sites of cytochrome P450 oxidation. A BDE < 85 kcal/mol is a flag for metabolic lability.
- **Isodesmic benchmarks**: The W4-11 and GMTKN55 datasets use thousands of isodesmic reactions to evaluate DFT methods. B3LYP/def2-TZVP achieves ~3 kcal/mol MAE on reaction energies.
- **Heats of formation**: The G4 and CBS-QB3 composite methods target < 1 kcal/mol accuracy for $\Delta H_f^\circ$, rivaling NIST experimental uncertainties.

## 📋 Summary

| Quantity | Formula | Typical DFT Error |
|---------|----------|------------------|
| $\Delta E_{rxn}$ | $\sum E_{prod} - \sum E_{react}$ | 2–5 kcal/mol (B3LYP) |
| $\Delta E_{rxn}^{ZPE}$ | $\Delta E_{rxn} + \Delta ZPE$ | 1–3 kcal/mol |
| BDE(A–B) | $E(A^\bullet) + E(B^\bullet) - E(AB)$ | 2–4 kcal/mol |
| Isodesmic $\Delta E$ | Same bond-type cancellation | < 1 kcal/mol |
| Atomization energy | $E_{mol} - \sum E_{atoms}$ | 5–20 kcal/mol (large errors) |
| Strain energy | $-\Delta E_{isodesmic}$ | 1–2 kcal/mol |

**Key insight**: Isodesmic reactions exploit systematic error cancellation; BDE calculations require unrestricted (open-shell) wavefunctions for radicals.

## 📝 Exercises

1. **Water formation**: The notebook computes $\Delta E$ for $2\text{H}_2 + \text{O}_2 \rightarrow 2\text{H}_2\text{O}$. Add a ZPE correction using approximate ZPEs (H₂: 6.2 kcal/mol, O₂: 2.3 kcal/mol, H₂O: 13.5 kcal/mol). Does agreement with experiment improve?

2. **BDE trend across halogens**: Compute BDE for HF, HCl, and HBr (use H–F: 0.917 Å, H–Cl: 1.275 Å, H–Br: 1.415 Å). Experimental BDEs: HF 136, HCl 103, HBr 87 kcal/mol. Plot computed vs experimental. What trend do you observe?

3. **Isodesmic for aromaticity**: Design an isodesmic reaction to estimate the resonance stabilization energy of benzene. One example: $\text{C}_6\text{H}_6 + 3\text{CH}_4 \rightarrow 3\text{C}_2\text{H}_4 + 3\text{C}_2\text{H}_2$ is *not* isodesmic. Find a reaction that *is* isodesmic and compute the stabilization.

4. **Basis set dependence**: Recompute BDE(H₂) with HF/STO-3G, HF/6-31G*, and B3LYP/def2-SVP. How does the basis set affect accuracy?

5. **Atomization energy**: Compute the atomization energy of CH₄ ($\text{CH}_4 \rightarrow \text{C} + 4\text{H}$). The experimental value is 398.8 kcal/mol. How does B3LYP/def2-SVP compare? Note that atomic energies require careful treatment of spin states (C: triplet, H: doublet).